In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # makes `src` importable from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats

from src.eda import (
    missing_value_summary, plot_missingno, plot_missingness_heatmap,
    plot_income_missing_distributions, run_little_mcar_test,
    point_biserial_missingness, plot_mar_evidence,
    plot_outlier_boxplots, iqr_outlier_summary,
    sentinel_value_audit, age_zero_check,
    plot_class_imbalance,
    plot_univariate_distributions, skewness_summary,
    plot_correlation_heatmap, bivariate_default_rate,
    plot_default_rate_by_decile,
    mann_whitney_tests, chi_squared_dependents,
)
from src.preprocessing import clean_train, clean_test
from src.features import engineer_features_train, engineer_features_test, prepare_model_inputs

train_df = pd.read_csv('../data/raw/cs-training.csv')
test_df  = pd.read_csv('../data/raw/cs-test.csv')

print(train_df.shape, test_df.shape)
train_df.head()

## 1. Exploratory Data Analysis

### 1.1 Missingness Audit

Identify which features have missing values, their exact counts and proportions,
and whether the missingness pattern carries information about the target.

In [ ]:
missing_value_summary(train_df)

In [ ]:
plot_missingno(train_df)

In [ ]:
plot_missingness_heatmap(train_df)

In [ ]:
plot_income_missing_distributions(train_df)

### Little's MCAR Test

* The only formal statistical test for MCAR.
* H₀: Data is Missing Completely At Random
* Reject H₀ if p < 0.05 → data is MAR or MNAR

In [ ]:
!pip install pyampute -q
run_little_mcar_test(train_df)

### Missingness Indicator Correlation (MAR Test)

Create a binary column `MonthlyIncome_missing` (1 = missing, 0 = observed) and
compute its point-biserial correlation with every continuous feature.

If significant correlations exist → missingness is predictable from observed features → MAR.

In [ ]:
corr_df = point_biserial_missingness(train_df)
corr_df

In [ ]:
plot_mar_evidence(train_df, corr_df)

### 1.2 Outlier & Sentinel Value Audit

In [ ]:
plot_outlier_boxplots(train_df)

In [ ]:
iqr_outlier_summary(train_df)

In [ ]:
sentinel_value_audit(train_df)

# The value 98/96 in delinquency columns is a sentinel placeholder — not a real
# observation (a borrower being 98 times late in 2 years is implausible).
# Treatment: replace with 0 and add a binary Sentinel_Flag.

In [ ]:
age_zero_check(train_df)

### 1.3 Class Imbalance Analysis

In [ ]:
plot_class_imbalance(train_df)

### 1.4 Univariate Distributions

In [ ]:
plot_univariate_distributions(train_df)

In [ ]:
skewness_summary(train_df)

### 1.5 Bivariate Analysis

In [ ]:
plot_correlation_heatmap(train_df)

In [ ]:
biv_df = bivariate_default_rate(train_df)
biv_df

**Top 4 Predictive Features** (by bivariate default-rate difference across the median):

| Feature | Default Rate above median | Default Rate below median | Difference |
|---|---|---|---|
| NumberOfTimes90DaysLate | 41.63% | 4.61% | **37.02 pp** |
| NumberOfTime60-89DaysPastDueNotWorse | 36.42% | 5.09% | **31.33 pp** |
| NumberOfTime30-59DaysPastDueNotWorse | 20.56% | 4.04% | **16.53 pp** |
| RevolvingUtilizationOfUnsecuredLines | 11.44% | 1.93% | **9.52 pp** |

In [ ]:
plot_default_rate_by_decile(train_df)

### 1.6 Statistical Tests

In [ ]:
mann_whitney_tests(train_df)

In [ ]:
chi_squared_dependents(train_df)

## 2. Data Cleaning & Imputation

All cleaning logic lives in `src/preprocessing.py`.  `clean_train()` fits
fittable objects (KNN imputer, mode) on the training set and returns them.
`clean_test()` accepts those fitted objects so no information leaks from test
into train.

Steps applied:
1. Flag + replace sentinel values (96, 98) in delinquency columns
2. Cap outliers: RevolvingUtilization → 2.0, DebtRatio → 5.0
3. Fix age == 0 → column median
4. Add `MonthlyIncome_missing` flag before imputation
5. Impute `NumberOfDependents` with training-set mode
6. Impute `MonthlyIncome` via KNN (k=5, fitted on training data)

In [ ]:
df_clean, income_imputer, mode_dep = clean_train(train_df)

print(f"Training set shape after cleaning: {df_clean.shape}")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")
df_clean.head()

### Imputation Comparison: Median vs KNN

KNN preserves the multivariate covariance structure of the data by
interpolating from the k nearest neighbours in feature space.
Median imputation collapses variance and introduces a spurious mode.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Compare variance preserved by each method on the raw training data
raw_var    = train_df['MonthlyIncome'].var()
median_var = train_df['MonthlyIncome'].fillna(train_df['MonthlyIncome'].median()).var()
knn_var    = df_clean['MonthlyIncome'].var()   # already KNN-imputed

print("--- MonthlyIncome Imputation Comparison ---")
print(f"Original Variance:       {raw_var:.0f}")
print(f"Median Imputed Variance: {median_var:.0f}")
print(f"KNN Imputed Variance:    {knn_var:.0f}")
print("Conclusion: Median artificially shrinks variance. KNN preserves data structure better.")

## 3. Feature Engineering

Four domain-informed features are constructed in `src/features.py`.

| Feature | Description |
|---|---|
| `Total_Late_Payments` | Sum of all three delinquency columns |
| `Income_Per_Dependent` | `MonthlyIncome / (NumberOfDependents + 1)` |
| `Financial_Stress_Index` | Correlation-weighted composite score |
| `Disposable_Income` | `MonthlyIncome × (1 − DebtRatio)`, clipped at 1st–99th pct |

FSI weights and Disposable_Income clip bounds are computed from the training set
and passed to the test pipeline — no leakage.

In [ ]:
df_clean, fsi_weights, disposable_bounds = engineer_features_train(df_clean)

print(f"Shape after feature engineering: {df_clean.shape}")
print(f"New columns: {['Total_Late_Payments','Income_Per_Dependent','Financial_Stress_Index','Disposable_Income']}")

### Scaling Strategy

**Tree-based models** — no scaling required.  Trees split on rank order, not
absolute magnitude.  The unscaled dataset retains all original delinquency
columns alongside `Total_Late_Payments` so the tree can use aggregate and
per-severity signals simultaneously.

**Linear models** — StandardScaler required for gradient descent stability.
The three original delinquency columns are also dropped to eliminate
multicollinearity with `Total_Late_Payments`.

In [ ]:
X_unscaled, X_scaled, y, scaler = prepare_model_inputs(df_clean)

## 4. Model Development

All models are evaluated with **5-fold stratified cross-validation** scored
on ROC-AUC and Macro-F1.  A shared `evaluate_model_cv()` helper handles SMOTE,
learning curves, PR curves, and confusion matrices.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, learning_curve, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_results = {}

def evaluate_model_cv(model, X, y, model_name, use_smote=True):
    print(f"\n{'='*50}")
    print(f"Evaluating {model_name} via 5-Fold Stratified CV")
    print(f"{'='*50}")

    from imblearn.pipeline import Pipeline as ImbPipeline
    pipeline = (
        ImbPipeline([('smote', SMOTE(random_state=42)), ('classifier', model)])
        if use_smote else model
    )

    scoring = {'roc_auc': 'roc_auc', 'macro_f1': 'f1_macro'}
    cv_results = cross_validate(pipeline, X, y, cv=skf, scoring=scoring,
                                return_train_score=False)

    auc_mean, auc_std = cv_results['test_roc_auc'].mean(), cv_results['test_roc_auc'].std()
    f1_mean,  f1_std  = cv_results['test_macro_f1'].mean(),  cv_results['test_macro_f1'].std()
    print(f"Mean CV ROC-AUC: {auc_mean:.4f} (+/- {auc_std:.4f})")
    print(f"Mean CV Macro-F1: {f1_mean:.4f} (+/- {f1_std:.4f})")
    model_results[model_name] = {'ROC-AUC': auc_mean, 'Macro-F1': f1_mean}

    # Learning curve
    train_sizes, train_scores, test_scores = learning_curve(
        pipeline, X, y, cv=skf, scoring='roc_auc', n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5)
    )
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_sizes, np.mean(train_scores, axis=1), label='Training AUC')
    plt.plot(train_sizes, np.mean(test_scores,  axis=1), label='CV AUC')
    plt.title(f'Learning Curve: {model_name}')
    plt.xlabel('Training Set Size'); plt.ylabel('ROC-AUC')
    plt.legend(); plt.grid(True)

    # PR curve on a single 80/20 split
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2,
                                                  stratify=y, random_state=42)
    if use_smote:
        X_tr, y_tr = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_val)[:, 1]
    preds = model.predict(X_val)
    precision, recall, _ = precision_recall_curve(y_val, probs)
    plt.subplot(1, 2, 2)
    plt.plot(recall, precision, marker='.', label=model_name)
    plt.title('Precision-Recall Curve')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()

    print("\nClassification Report (Validation Set):")
    print(classification_report(y_val, preds))
    print("Confusion Matrix:")
    sns.heatmap(confusion_matrix(y_val, preds), annot=True, fmt='d', cmap='Blues')
    plt.show()

### Handling Class Imbalance: SMOTE vs scale_pos_weight

The dataset has severe class imbalance — only 6.68% defaults.  Two strategies
are compared head-to-head:

* **SMOTE**: synthesises new minority-class examples by interpolating between
  existing high-risk borrowers in feature space → richer decision boundaries.
* **`scale_pos_weight`** (XGBoost native): up-weights the loss on minority-class
  errors during training.  No synthetic rows, cleaner signal.

### Model 1: Logistic Regression (baseline)

Assumes a linear relationship between log-odds of default and features.
The three original delinquency columns are dropped (multicollinearity with
`Total_Late_Payments`) so the independence assumption holds.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
evaluate_model_cv(lr_model, X_scaled, y, "Logistic Regression (SMOTE)", use_smote=True)

In [ ]:
lr_model_no_smote = LogisticRegression(max_iter=1000, random_state=42)
evaluate_model_cv(lr_model_no_smote, X_scaled, y,
                  "Logistic Regression (No SMOTE)", use_smote=False)

### Model 2: Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
evaluate_model_cv(dt_model, X_unscaled, y, "Decision Tree", use_smote=True)

### Model 3: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                   random_state=42, n_jobs=-1)
evaluate_model_cv(rf_model, X_unscaled, y, "Random Forest", use_smote=True)

### Model 4: XGBoost — Head-to-Head Comparison

Two XGBoost variants tuned via `RandomizedSearchCV` (5 iterations, 5-fold CV):

* **4A**: `scale_pos_weight` = ratio of negative to positive class samples
* **4B**: SMOTE + L1/L2 regularisation (`reg_alpha`, `reg_lambda`) to suppress
  noise introduced by synthetic samples

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

print("Starting XGBoost Head-to-Head Comparison...")

# --- 4A: scale_pos_weight ---
scale_weight = len(y[y==0]) / len(y[y==1])
xgb_weight = XGBClassifier(scale_pos_weight=scale_weight,
                            eval_metric='logloss', random_state=42)
param_dist_weight = {
    'max_depth':       [3, 5, 7],
    'learning_rate':   [0.01, 0.05, 0.1],
    'n_estimators':    [100, 200, 300],
    'subsample':       [0.8, 1.0],
    'colsample_bytree':[0.8, 1.0],
}
search_weight = RandomizedSearchCV(xgb_weight, param_distributions=param_dist_weight,
                                   n_iter=5, cv=skf, scoring='roc_auc',
                                   random_state=42, n_jobs=-1)
search_weight.fit(X_unscaled, y)
best_xgb_weight = search_weight.best_estimator_
evaluate_model_cv(best_xgb_weight, X_unscaled, y,
                  "XGBoost (Scale Weight)", use_smote=False)

# --- 4B: SMOTE + L1/L2 ---
tuning_pipeline_smote = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(eval_metric='logloss', random_state=42))
])
param_dist_smote = {
    'classifier__max_depth':       [3, 5],
    'classifier__learning_rate':   [0.01, 0.05],
    'classifier__n_estimators':    [100, 200, 300],
    'classifier__subsample':       [0.6, 0.8],
    'classifier__colsample_bytree':[0.6, 0.8],
    'classifier__reg_alpha':       [0, 1, 5],
    'classifier__reg_lambda':      [1, 5, 10],
}
search_smote = RandomizedSearchCV(tuning_pipeline_smote,
                                  param_distributions=param_dist_smote,
                                  n_iter=5, cv=skf, scoring='roc_auc',
                                  random_state=42, n_jobs=-1)
search_smote.fit(X_unscaled, y)
best_xgb_smote = search_smote.best_estimator_.named_steps['classifier']
print(f"Best SMOTE Params: {search_smote.best_params_}")
evaluate_model_cv(best_xgb_smote, X_unscaled, y,
                  "XGBoost (SMOTE + Reg)", use_smote=True)

## 5. Model Comparison

In [ ]:
final_results_data = {
    'Logistic Regression (No SMOTE)': {
        'ROC-AUC (CV)': 0.8488, 'Macro-F1 (CV)': 0.6119,
        'Precision (Class 1)': 0.60, 'Recall (Class 1)': 0.17, 'Accuracy': 0.94,
    },
    'Logistic Regression (SMOTE)': {
        'ROC-AUC (CV)': 0.8519, 'Macro-F1 (CV)': 0.6062,
        'Precision (Class 1)': 0.21, 'Recall (Class 1)': 0.75, 'Accuracy': 0.80,
    },
    'Decision Tree (SMOTE)': {
        'ROC-AUC (CV)': 0.8092, 'Macro-F1 (CV)': 0.6442,
        'Precision (Class 1)': 0.27, 'Recall (Class 1)': 0.56, 'Accuracy': 0.87,
    },
    'Random Forest (SMOTE)': {
        'ROC-AUC (CV)': 0.8517, 'Macro-F1 (CV)': 0.6467,
        'Precision (Class 1)': 0.26, 'Recall (Class 1)': 0.66, 'Accuracy': 0.85,
    },
    'XGBoost (SMOTE)': {
        'ROC-AUC (CV)': 0.8535, 'Macro-F1 (CV)': 0.6276,
        'Precision (Class 1)': 0.24, 'Recall (Class 1)': 0.72, 'Accuracy': 0.83,
    },
    'XGBoost (scale_pos_weight)': {
        'ROC-AUC (CV)': 0.8644, 'Macro-F1 (CV)': 0.6056,
        'Precision (Class 1)': 0.21, 'Recall (Class 1)': 0.78, 'Accuracy': 0.79,
    },
}

comparison_df = pd.DataFrame(final_results_data).T.sort_values('ROC-AUC (CV)', ascending=False)
print("=" * 80)
print("FINAL UNIFIED MODEL COMPARISON TABLE")
print("=" * 80)
display(comparison_df.style.highlight_max(
    subset=['ROC-AUC (CV)', 'Recall (Class 1)'], color='lightgreen'
))

## 6. Feature Importance

In [ ]:
# Re-fit LR on SMOTE-resampled 80% split so we have a fitted model for coefficients
X_tr_lr, _, y_tr_lr, _ = train_test_split(X_scaled, y, test_size=0.2,
                                           stratify=y, random_state=42)
X_tr_lr_sm, y_tr_lr_sm = SMOTE(random_state=42).fit_resample(X_tr_lr, y_tr_lr)
lr_model.fit(X_tr_lr_sm, y_tr_lr_sm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost feature importance
xgb_imp = best_xgb_weight.feature_importances_
xgb_idx = np.argsort(xgb_imp)[::-1]
sns.barplot(x=xgb_imp[xgb_idx][:10],
            y=[X_unscaled.columns[i] for i in xgb_idx][:10],
            ax=axes[0], palette='Reds_r')
axes[0].set_title('Top 10 XGBoost Features (scale_pos_weight)', fontweight='bold')
axes[0].set_xlabel('Relative Importance')

# Logistic Regression coefficients
lr_coefs = lr_model.coef_[0]
lr_idx   = np.argsort(np.abs(lr_coefs))[::-1]
lr_vals  = lr_coefs[lr_idx]
colors   = ['#d62728' if v > 0 else '#2ca02c' for v in lr_vals[:10]]
sns.barplot(x=lr_vals[:10],
            y=[X_scaled.columns[i] for i in lr_idx][:10],
            ax=axes[1], palette=colors)
axes[1].set_title('Top 10 LR Coefficients (Red = ↑ risk, Green = ↓ risk)', fontweight='bold')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

## 7. Decision Threshold Analysis

In [ ]:
_, X_holdout, _, y_holdout = train_test_split(X_unscaled, y,
                                               test_size=0.2, stratify=y,
                                               random_state=99)
probs = best_xgb_weight.predict_proba(X_holdout)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_holdout, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

optimal_idx       = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
safe_idx          = np.abs(recall - 0.90).argmin()
safe_threshold    = thresholds[safe_idx]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision[:-1], label='Precision', color='blue', lw=2)
plt.plot(thresholds, recall[:-1],    label='Recall',    color='green', lw=2)
plt.plot(thresholds, f1_scores[:-1], label='F1-Score',  color='red', lw=2, linestyle='--')
plt.axvline(optimal_threshold, color='black', linestyle=':',
            label=f'Max F1 ({optimal_threshold:.2f})')
plt.axvline(safe_threshold, color='purple', linestyle=':',
            label=f'90% Recall threshold ({safe_threshold:.2f})')
plt.title('Metrics vs Decision Threshold', fontweight='bold')
plt.xlabel('Probability Threshold'); plt.ylabel('Score')
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 8. Test Set Prediction

Apply identical cleaning and feature engineering to the test set using
the fitted artefacts from the training pipeline — no information leaks.

In [ ]:
# Clean test set using training-fitted artefacts
test_clean = clean_test(test_df, income_imputer, mode_dep)

# Engineer features using training-fitted artefacts
test_clean = engineer_features_test(test_clean, fsi_weights, disposable_bounds)

# Align columns to the exact order seen during training
X_test_final = test_clean.drop(['SeriousDlqin2yrs', 'Id'], axis=1, errors='ignore')
X_test_final = X_test_final[X_unscaled.columns]

# Generate probabilities from the winning model
final_probabilities = best_xgb_weight.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    'Id':          test_clean['Id'],
    'Probability': final_probabilities,
})
submission.to_csv('../data/processed/predictions.csv', index=False)
print("Submission saved to data/processed/predictions.csv")
display(submission)